# Step 2 — Ring-geometry inference with **emcee** (MCMC)

**Goal.** The same inference as
[`02_inference_dynesty.ipynb`](02_inference_dynesty.ipynb) — the same forward model, the same
KDE likelihood, the same priors — sampled with a different algorithm.

## Why run it twice

The likelihood here is a kernel density estimate over a covariant set of observables, which
gives it an irregular shape: correlated ridges, and in some configurations multiple modes. Any
single sampler can misrepresent such a surface, so agreement between two independent samplers
is evidence that the posterior is a property of the *problem* rather than of the algorithm.
Where they disagree, the posterior should not be trusted.

The division of labour:

- **`dynesty`** (nested sampling) is the primary sampler. It explores from the prior inwards,
  handles multimodality naturally, and returns the Bayesian evidence $\ln Z$ — which is what
  makes different configurations comparable rather than merely each fitted.
- **`emcee`** (affine-invariant ensemble MCMC) is the cross-check. It gives no evidence
  estimate, and it needs its convergence assessed explicitly (§7) rather than terminating on
  a principled stopping rule.

For the model, the likelihood and the priors, see §1–§2 below — they are the same text as in the
dynesty notebook, deliberately.

## What differs in practice

| | `dynesty` | `emcee` |
|---|---|---|
| Configuration | `NS_CONFIG` | `MCMC_CONFIG` |
| Termination | remaining-evidence criterion `dlogz` | fixed number of steps |
| Evidence $\ln Z$ | yes | no |
| Convergence | built into the stopping rule | assessed post-hoc: log-prob trace + autocorrelation time |
| Post-processing | none | discard burn-in, then thin |


## 0. Environment

In [ ]:
# ── Bootstrap: make the sibling packages importable without installation ────
# The pipeline uses three packages that live in the repository, uninstalled:
#   exorings, geotrans   (repo root)      photoring   (pipeline/)
# We locate the repo root and pipeline/ robustly from the current working dir
# (Jupyter / nbconvert / papermill all run notebooks from pipeline/).
import sys, pathlib
_HERE   = pathlib.Path.cwd().resolve()
_cands  = [_HERE, *_HERE.parents]
_NB_DIR = next((c for c in _cands if (c / "photoring").is_dir()), _HERE)
_REPO   = next((c for c in _cands if (c / "exorings").is_dir()), _NB_DIR.parent)
for _p in (str(_REPO), str(_NB_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo root :", _REPO)
print("pipeline  :", _NB_DIR)

In [ ]:
import numpy as np
import warnings, os, time
warnings.filterwarnings("ignore")
# Limit BLAS threads before heavy numerics (pool workers each add threads).
for _v in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_v, "2")

import photoring as pr
import photoring.plotting as plot
plot.apply_style()

import emcee
print("emcee", emcee.__version__)

## 1. USER CONFIGURATION  ← edit here (papermill-injected)

This cell carries the `parameters` tag, so `run_sweep.py` can override any of it to sweep
configurations. Four groups:

- **`CASE` / `PLANET`** — which case directory and which planet.
- **`PLANET_PARAMS`** — per-planet external constraints: the impact-parameter prior centre and
  width, and the reference radius ratio with its lower prior bound as a fraction.
- **`KDE_CONFIG`** — which observables enter the likelihood, how many posterior samples train
  the KDE, and the training seed.
- **`NS_CONFIG`** — nested-sampling settings.
- **`MODEL_CONFIG`** — which parameters are free and which are fixed (see §2).

### Priors, and the reasoning behind each

| Parameter | Prior | Why |
|---|---|---|
| $f_e$ | $\mathcal{U}(1,\ f_{e,\max})$ | $f_{e,\max}=10$ from the prograde stability limit near $0.5\,R_{\rm Hill}$ |
| $f_i$ | fixed at 1 | the photometry cannot resolve structure interior to the ring's outer edge |
| $i_R$ | $p(i_R)\propto\sin i_R$ | isotropic orientations; note this already favours edge-on |
| $\theta_R$ | $\mathcal{U}(0^\circ,90^\circ)$ | no preferred azimuth |
| $p$ | $\mathcal{U}(p_{\min},\ p_{\rm obs})$ | $p_{\min}$ from a maximum, Earth-like bulk density; the upper bound is the ringless fit |
| $\tau$ | $p(\tau)\propto 1/\tau$ on $[0.1,10]$ | log-uniform over an unknown scale; fixed to 1 in baseline runs, comparable to Saturn's main rings, to avoid a parameter partly degenerate with ring size and orientation |
| $\rho_{\star,\rm true}$ | empirical KDE of an independent (isochrone) posterior | an external constraint, not something the transit should revise |
| $b$ | truncated normal on $[0,1]$ | centred on the published ringless fit; bounded because $b$ is physically confined |

Letting $p$ float matters physically: part of the observed depth may be produced by the ring,
which implies a *smaller* planet. Fixing $p = p_{\min}$ instead explores the opposite limit —
maximum ring extent — and is best read as a bounding case rather than a favoured solution.


In [ ]:
CASE   = "kepler_51"
PLANET = "d"

PLANET_PARAMS = {
    "d": dict(B_FIXED=0.0030, B_SIGMA=2 * 0.0950, p_mean_ref=0.09857, p_prior_lo=0.20353),
    "b": dict(B_FIXED=0.0740, B_SIGMA=2 * 0.0720, p_mean_ref=0.07225, p_prior_lo=0.27767),
}

KDE_CONFIG = {"observables": ["delta", "rho_obs", "T14"], "N_KDE": 5000, "seed_kde": 123}

MCMC_CONFIG = {"nwalkers": 64, "nsteps": 10000, "burnin": 2000, "thin": 50,
               "seed": 2026, "use_pool": True, "n_procs": 3}

# Defaults reproduce the manuscript's adopted configuration: the *fully free*
# model, which is the only one in the retrieval suite that reproduces the full set
# of transit observables (see section 2). Flip any flag to False to explore the
# simpler variants; run_sweep.py sweeps them.
MODEL_CONFIG = {
    "B_FREE": True, "B_FIXED": PLANET_PARAMS[PLANET]["B_FIXED"],
    "B_SIGMA": PLANET_PARAMS[PLANET]["B_SIGMA"],
    "RHO_TRUE_FREE": True, "RHO_TRUE_FIXED": None,
    "FI_FIXED": 1.0, "FE_MAX": 10.0,
    "TAU_FREE": True, "TAU_FIXED": 1.0, "TAU_PRIOR_LO": 0.1, "TAU_PRIOR_HI": 10.0,
    "P_FREE": True, "p_mean_ref": PLANET_PARAMS[PLANET]["p_mean_ref"],
    "p_prior_lo": PLANET_PARAMS[PLANET]["p_prior_lo"], "p_prior_hi": 1.0,
    "FORWARD_MODEL": "exorings",
}

## 2. Build the model (data + KDE likelihood + priors)

`PhotoRingModel` assembles the forward model, the KDE likelihood and the prior transform, and
resolves which parameters are free.

### Free vs fixed, and why it changes the answer

$b$ and $\rho_{\star,\rm true}$ are not ring properties — they set the transit chord and the
orbital scale — but they carry observational uncertainty. Three treatments are possible: fix
them to their central values, sample them under informative priors, or marginalise over them
inside the likelihood via a pseudo-marginal Monte Carlo average

$$\mathcal{L}(\boldsymbol\theta) \approx \frac{1}{M}\sum_{j=1}^{M}
  \widehat{p}\!\left(\mathbf{y} = \mathbf{y}\big(\boldsymbol\theta,\
  \rho_{\star,\rm true}^{(j)},\ b^{(j)}\big)\right),$$

with the draws taken from their adopted priors. The last is the formally correct Bayesian
treatment when these are *external* constraints that the transit data should not revise.

This choice is consequential, not a detail: fixing $\rho_{\star,\rm true}$ forces the ring
geometry to absorb all the tension and the model then fails to reproduce the transit durations.
Sampling both jointly is what recovers a geometry consistent with the full observable set.

The dimensionality is $4 + $ `TAU_FREE` $+$ `RHO_TRUE_FREE` $+$ `B_FREE` (4–7D), and the
parameter vector order is always
`[fe, ir, theta, p, (tau), (rho_true), (b)]`.


In [ ]:
paths = pr.CasePaths(CASE)
FORWARD_MODEL = str(MODEL_CONFIG.get("FORWARD_MODEL", "exorings")).lower()
paths.ensure_outputs(FORWARD_MODEL)

# Load the case's derived observables, rho_true samples and inverse-CDF grid.
data  = pr.load_case_data(paths, PLANET)
model = pr.PhotoRingModel(
    data["ttv"], data["rho_true_gcc_samples"], MODEL_CONFIG, KDE_CONFIG,
    rho_grid=data["rho_grid"], rho_cdf=data["rho_cdf"], p_fixed=data["P_fixed"],
)
print(f"Planet {PLANET}: {len(data['ttv']['delta'])} TTV samples | P_fixed={data['P_fixed']:.6f} d")
print(f"NDIM={model.NDIM}  params={model.PARAM_NAMES}")

In [ ]:
_kt = "-".join(model.observables)
RUN_TAG = (f"{CASE}_{PLANET}_MCMC_{FORWARD_MODEL}_kde_{_kt}"
           f"_nw{MCMC_CONFIG['nwalkers']}_ns{MCMC_CONFIG['nsteps']}"
           f"_bi{MCMC_CONFIG['burnin']}_th{MCMC_CONFIG['thin']}"
           f"_NKDE{KDE_CONFIG['N_KDE']}_seed{MCMC_CONFIG['seed']}{model.free_tag()}")
print("RUN_TAG:", RUN_TAG)

## 3. KDE self-consistency check

Before sampling: does the KDE actually represent the posterior it was trained on? This draws
fresh samples from the fitted KDE and overlays them on the training histograms. A visible
mismatch here means the likelihood target is wrong, and everything downstream would inherit
that — a bandwidth or a training-sample problem is far cheaper to catch now than after a run.


In [ ]:
import matplotlib.pyplot as plt
plot.plot_kde_ppc(model, planet=PLANET, paths=paths, run_tag=RUN_TAG); plt.show()

## 4. Run the MCMC

Walkers are initialised by drawing from the prior and rejecting any draw with $-\infty$
log-probability, so the ensemble starts inside the physically valid region.

The move set mixes differential-evolution proposals, which handle the correlated ridges of this
likelihood better than a plain stretch move — the $f_e$–$p$ and $i_R$–$\theta_R$ degeneracies
(§8) are exactly the geometry that defeats axis-aligned proposals.


In [ ]:
try:    import multiprocess as mp
except ImportError: import multiprocessing as mp
try:    _ctx = mp.get_context("fork")
except (AttributeError, ValueError): _ctx = mp

result = pr.run_emcee(model, MCMC_CONFIG, ctx=_ctx)
print(f"\nacc frac = {result['acc_frac']*100:.1f}%  | runtime {result['runtime_s']:.1f}s | N={len(result['chain'])}")

## 5. Posterior summary

Median and 68% credible interval per parameter, computed **after** burn-in removal and thinning.

Compare these against the dynesty run for the same configuration: consistent medians and widths
are the cross-check this notebook exists for. A disagreement larger than the quoted intervals
points to one of the samplers having failed to explore the posterior, not to a real difference.


In [ ]:
for name in model.PARAM_NAMES:
    s = result["stats"][name]; m = s["median"]
    print(f"  {name:>10}: {m:.5f}  [-{m-s['p16']:.5f}, +{s['p84']-m:.5f}]")

## 6. Posterior predictive check (all observables)

Posterior samples are pushed back through the forward model and the predicted distribution of
each observable is compared against the TTV-derived one, with three statistics: the
1-Wasserstein distance $W_1$ (mean absolute displacement between the empirical distributions),
the Kolmogorov–Smirnov $p$-value $p_{\rm KS}$ (null: same parent distribution), and an energy
distance $E$ (squared discrepancy).

**All five observables are checked, including those left out of the likelihood.** Fitting an
observable does not guarantee the others follow, so the excluded ones are a genuine
out-of-sample test of the inferred geometry. Panel titles mark which is which.


In [ ]:
import matplotlib.pyplot as plt
ppc = pr.compute_ppc(model, result["chain"])
run = pr.make_run(model, result, RUN_TAG, PLANET, ppc=ppc)
plot.plot_ppc(run, data["ttv"], paths=paths); plt.show()

## 7. emcee native diagnostics (log-prob trace + autocorrelation)

Unlike nested sampling, an MCMC run has no principled stopping rule, so convergence has to be
demonstrated. Two diagnostics:

- **Median log-posterior against step number.** It should rise and then plateau. Where it
  plateaus is where burn-in ends: samples before that point are still approaching the typical
  set and must be discarded.
- **Integrated autocorrelation time $\hat\tau$ per parameter.** How many steps the chain needs
  to produce one effectively independent sample. The chain should be many multiples of
  $\hat\tau$ long (a common rule of thumb is $\gtrsim 50\,\hat\tau$), and thinning by roughly
  $\hat\tau$ turns the chain into near-independent draws. A parameter with a much larger
  $\hat\tau$ than the rest is the one the sampler is struggling with — usually one caught in a
  degeneracy.

Both are diagnostics of the *sampler*, not of the science. If they look bad, increase `nsteps`
or `nwalkers` in §1 and rerun before reading anything into the posterior.


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
_med = np.median(result["logprob_raw"], axis=1)
axes[0].plot(_med, lw=0.9); axes[0].axvline(MCMC_CONFIG["burnin"], color="k", ls="--", label="burn-in")
axes[0].set_xlabel("step"); axes[0].set_ylabel("median log-prob"); axes[0].legend(fontsize=8)
try:
    import emcee
    tau = emcee.autocorr.integrated_time(result["chain_raw"], tol=0)
    axes[1].bar(model.PARAM_NAMES, tau, color=plot.planet_color(PLANET), alpha=0.8)
    axes[1].set_ylabel(r"$\hat{\tau}$ (integrated autocorr.)")
    print("autocorr times:", dict(zip(model.PARAM_NAMES, np.round(tau, 1))))
except Exception as e:
    axes[1].set_axis_off(); print("autocorr failed:", e)
fig.tight_layout()
fig.savefig(paths.figures_dir("diagnostics") / f"{RUN_TAG}_diagnostics.png", dpi=plot.STYLE["fig_dpi"]); plt.show()

## 8. Marginals and corner (publication style)

The 1-D marginals with their priors overlaid, and the joint posterior as a corner plot.

Note that the corner is drawn by `dynesty`'s plotting routine, which needs the weighted
nested-sampling output — so for an emcee run it is unavailable and the call is skipped. Use the
marginals here, and the dynesty notebook for the corner.

The degeneracies to look for are the structural ones: $f_e$ against $p$ (any pair preserving the
effective occulting area gives the same transit depth) and $i_R$ against $\theta_R$ (the same
projected ellipse from different orientation pairs).


In [ ]:
import matplotlib.pyplot as plt
plot.plot_marginals(run, berger_rho=data["rho_true_gcc_samples"], paths=paths); plt.show()
plot.plot_corner(run, paths=paths); plt.show()

## 9. Save results

Writes `<RUN_TAG>.npz` (the thinned chain, the raw chain, log-probabilities and the PPC array)
plus `<RUN_TAG>_meta.json` (every configuration value, the acceptance fraction and the derived
summary statistics) into `results/<forward_model>/`.

The tag begins `MCMC_` rather than `NS_`, so emcee and dynesty runs of the same configuration
sit side by side in one results directory without colliding, and step 3 discovers both.


In [ ]:
meta = dict(
    planet=PLANET, case=CASE, run_tag=RUN_TAG, sampler="emcee",
    kde_observables=model.observables, N_KDE=int(len(model.idx_train)),
    seed_kde=int(KDE_CONFIG["seed_kde"]),
    nwalkers=int(MCMC_CONFIG["nwalkers"]), nsteps=int(MCMC_CONFIG["nsteps"]),
    burnin=int(MCMC_CONFIG["burnin"]), thin=int(MCMC_CONFIG["thin"]),
    seed_mcmc=int(MCMC_CONFIG["seed"]), FORWARD_MODEL=FORWARD_MODEL,
    B_FREE=bool(model.B_FREE), B_FIXED=float(model.B_FIXED), B_SIGMA=float(model.B_SIGMA),
    RHO_TRUE_FREE=bool(model.RHO_TRUE_FREE), RHO_TRUE_FIXED=float(model.RHO_TRUE_FIXED),
    TAU_FREE=bool(model.TAU_FREE), TAU_FIXED=float(model.TAU_FIXED),
    P_FREE=bool(model.P_FREE), FI_FIXED=float(model.FI_FIXED), FE_MAX=float(model.FE_MAX),
    p_min=float(model.p_min), p_max=float(model.p_max), p_mean_ref=float(model.p_mean_ref),
    P_fixed_days=float(model.P_fixed),
    acc_frac=float(result["acc_frac"]), runtime_s=float(result["runtime_s"]),
    n_samples=int(len(result["chain"])), param_names=model.PARAM_NAMES,
)
for _n, _s in result["stats"].items():
    meta[f"stat_{_n}_median"] = float(_s["median"]); meta[f"stat_{_n}_p16"] = float(_s["p16"]); meta[f"stat_{_n}_p84"] = float(_s["p84"])

arrays = dict(chain=result["chain"], logprob=result["logprob"],
              chain_raw=result["chain_raw"], logprob_raw=result["logprob_raw"], ppc=ppc)
pr.save_run(paths.results_dir(FORWARD_MODEL), RUN_TAG, arrays, meta)
print("Saved ->", paths.results_dir(FORWARD_MODEL))